# HelloCret — English script to vertical video

Choose a Colab T4 GPU runtime, run the setup cell and then the interface cell. The only per-video input is an English script. After you click **Generate vertical video**, Chatterbox narration, scene rendering and captioning run in this notebook and produce a 720×1280 MP4.

The scenes are locally drawn procedural cartoons, not generated 3D footage. Word timing is estimated per spoken sentence and is not phoneme-level forced alignment. The model's default speaker is consistent as a model choice but is not guaranteed to be a US male; a fixed voice identity/accent requires a licensed reference WAV configured once in the setup cell.


In [ ]:
import subprocess, sys
from pathlib import Path
ROOT = Path('/content/hellocret')
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/tred13/hellocret.git', str(ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'chatterbox-tts', 'pillow'], check=True)
sys.path.insert(0, str(ROOT / 'src'))
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab before generating.')
print('GPU:', torch.cuda.get_device_name(0))
print('Project ready.')


In [ ]:
from pathlib import Path
import torch, torchaudio, ipywidgets as widgets
from IPython.display import Video, FileLink, display, clear_output
from hellocret.pipeline import caption_schedule_segments, render_video, split_beats, validate_script
VOICE_REFERENCE_PATH = None  # optional one-time licensed speaker WAV
PAUSE_SECONDS = 0.12
OUTPUT_DIR = Path('/content/hellocret/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
_MODEL = None
def get_model():
    global _MODEL
    if _MODEL is None:
        from chatterbox.tts import ChatterboxTTS
        _MODEL = ChatterboxTTS.from_pretrained(device='cuda')
    return _MODEL
def synthesize_script(script, output_audio):
    model = get_model(); sentences = split_beats(script); chunks = []; durations = []
    for index, sentence in enumerate(sentences):
        if VOICE_REFERENCE_PATH and Path(VOICE_REFERENCE_PATH).is_file():
            wav = model.generate(sentence, audio_prompt_path=VOICE_REFERENCE_PATH)
        else:
            wav = model.generate(sentence)
        wav = wav.detach().to('cpu')
        if wav.ndim == 1: wav = wav.unsqueeze(0)
        durations.append(wav.shape[-1] / model.sr); chunks.append(wav)
        if index < len(sentences) - 1:
            chunks.append(torch.zeros((1, int(model.sr * PAUSE_SECONDS)), dtype=wav.dtype))
        print(f'Narration {index + 1}/{len(sentences)} ready')
    audio = torch.cat(chunks, dim=-1); torchaudio.save(str(output_audio), audio, model.sr)
    captions, expected = caption_schedule_segments(script, durations, PAUSE_SECONDS)
    if abs(expected - audio.shape[-1] / model.sr) > 0.05: raise RuntimeError('Narration timing mismatch.')
    return captions, durations
def generate_video(_):
    generate_button.disabled = True
    with result_box:
        clear_output()
        try:
            script = validate_script(script_box.value)
            audio_path = OUTPUT_DIR / 'narration.wav'; video_path = OUTPUT_DIR / 'hellocret_vertical.mp4'
            print('Generating narration with Chatterbox...')
            captions, segment_durations = synthesize_script(script, audio_path)
            progress = widgets.IntProgress(value=0, min=0, max=100, description='Rendering'); display(progress)
            def on_progress(done, total): progress.value = int(done * 100 / total)
            meta = render_video(script, audio_path, video_path, progress=on_progress, captions=captions, segment_durations=segment_durations, pause_seconds=PAUSE_SECONDS)
            print('Finished:', meta['width'], 'x', meta['height'], 'at', meta['fps'], 'fps')
            display(Video(str(video_path), embed=True, width=320)); display(FileLink(str(video_path)))
        except Exception as exc:
            print('Generation stopped:', str(exc))
        finally:
            generate_button.disabled = False
script_box = widgets.Textarea(value='', description='English script only', placeholder='Paste the English narration script here...', layout=widgets.Layout(width='100%', height='180px'))
generate_button = widgets.Button(description='Generate vertical video', button_style='primary')
result_box = widgets.Output(); generate_button.on_click(generate_video)
display(script_box, generate_button, result_box)
